In [ ]:
# -*- coding: utf-8 -*-
"""
FINAL 5-FOLD EXPERIMENT-GROUPED × 10-SEED H=10 ABLATION STUDY
=============================================================

Purpose
-------
Repeat the corrected experiment-grouped H=20 ablation protocol at H=10,
while keeping the historical input window fixed at 20 frames.

Configurations
--------------
1. Sensor-only LSTM                  : 8 features
2. Vision-only LSTM                  : 12 features
3. Fusion-LSTM without derivatives  : 10 features
4. Fusion-LSTM with derivatives     : 20 features

Evaluation protocol
-------------------
- Historical input window = 20 frames for every configuration.
- Forecasting horizon H = 10 frames (~0.333 s at 30 FPS).
- Same five outer experiment-grouped folds for all configurations.
- Outer test folds exactly match the successful horizon-selection GroupKFold protocol.
- Complete welding experiments are held out for testing.
- Two complete validation experiments are chosen from development data only.
- Validation-pair selection is four-class-coverage aware.
- No experiment overlap among train / validation / test.
- Sequences never cross welding-experiment boundaries.
- First-order differences are recalculated within each experiment.
- StandardScaler is fitted on training sequences only.
- Class weights are calculated from training targets only.
- Weighted cross-entropy is used.
- The checkpoint with the lowest validation loss is selected.
- Test folds are evaluated only after model selection.
- Seeds 42..51 are used identically for every configuration.
- Primary comparison uses pooled out-of-fold predictions over all experiments.

Expected model runs
-------------------
4 configurations × 5 folds × 10 seeds = 200 models
"""

from __future__ import annotations

import copy
import itertools
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset


# =============================================================================
# 1. PATHS
# =============================================================================

PROJECT_DIR = Path.cwd()

DATASET_PATH = (
    PROJECT_DIR
    / "outputs"
    / "master_fusion_dataset_clean.csv"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "outputs"
    / "Final_Grouped_H10_Ablation_HorizonFolds_5Fold_10Seeds"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 2. STUDY SETTINGS
# =============================================================================

SEQUENCE_LENGTH = 20
FUTURE_HORIZON = 10
FPS = 30.0

SEEDS = list(range(42, 52))  # 42, ..., 51

NUM_CLASSES = 4
CLASS_NAMES = [
    "Good",
    "Burr",
    "Flash-burr",
    "Surface-groove/void",
]

EPOCHS = 60
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
HIDDEN_SIZE = 64
NUM_LAYERS = 2
LSTM_DROPOUT = 0.2
FC_DROPOUT = 0.2

NUM_WORKERS = 0  # safest setting on Windows
SAVE_MODEL_CHECKPOINTS = False

# Canonical outer test folds from the successful horizon-selection script.
# These are the exact 5 GroupKFold test assignments generated once from
# sorted experiment IDs and reused for H = 5, 10, 15, 20, 25, 30.
# Keeping these exact folds is essential for a direct H=10 comparison.
OUTER_TEST_FOLDS = [
    ["exp_1", "exp_4", "exp_9"],
    ["exp_3", "exp_8"],
    ["exp_2", "exp_7"],
    ["exp_11", "exp_6"],
    ["exp_10", "exp_5"],
]

EXPECTED_INTERNAL_EXPERIMENTS = sorted(
    {exp for fold in OUTER_TEST_FOLDS for exp in fold}
)

EXPECTED_SEQUENCE_COUNT = 2581


# =============================================================================
# 3. FEATURE DEFINITIONS
# =============================================================================

SENSOR_RAW_FEATURES = [
    "sensor_force",
    "sensor_rpm",
    "sensor_torque",
    "sensor_temp",
]

VISION_RAW_FEATURES = [
    "weld_width_mm",
    "weld_area_mm2",
    "Burrs_area_mm2",
    "flash_burr_area_mm2",
    "surface_groove_void_area_mm2",
    "total_defect_area_mm2",
]

RAW_FEATURES = SENSOR_RAW_FEATURES + VISION_RAW_FEATURES

SENSOR_DIFF_FEATURES = [
    f"{name}_diff" for name in SENSOR_RAW_FEATURES
]

VISION_DIFF_FEATURES = [
    f"{name}_diff" for name in VISION_RAW_FEATURES
]

DIFF_FEATURES = SENSOR_DIFF_FEATURES + VISION_DIFF_FEATURES

FULL_FEATURES = RAW_FEATURES + DIFF_FEATURES

CONFIGURATIONS = {
    "Sensor-only LSTM": (
        SENSOR_RAW_FEATURES
        + SENSOR_DIFF_FEATURES
    ),
    "Vision-only LSTM": (
        VISION_RAW_FEATURES
        + VISION_DIFF_FEATURES
    ),
    "Fusion-LSTM without derivatives": RAW_FEATURES,
    "Fusion-LSTM with derivatives": FULL_FEATURES,
}


# =============================================================================
# 4. REPRODUCIBILITY
# =============================================================================

def set_global_seed(seed: int) -> None:
    """Set all relevant random seeds."""
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Reproducibility is preferred here because the study explicitly compares
    # the same numbered seeds across ablation configurations.
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


# =============================================================================
# 5. LABEL HANDLING
# =============================================================================

def normalize_class_label(value) -> int:
    """
    Robustly map defect_class to:
      0 = Good
      1 = Burr
      2 = Flash-burr
      3 = Surface-groove/void
    """
    if pd.isna(value):
        raise ValueError("Found missing value in the frame-level class label.")

    # Numeric class IDs.
    if isinstance(value, (int, np.integer)):
        label = int(value)
        if label in (0, 1, 2, 3):
            return label

    if isinstance(value, (float, np.floating)):
        if float(value).is_integer():
            label = int(value)
            if label in (0, 1, 2, 3):
                return label

    text = str(value).strip().lower()
    text = (
        text.replace("-", "_")
        .replace("/", "_")
        .replace(" ", "_")
    )

    aliases = {
        "0": 0,
        "good": 0,
        "good_weld": 0,

        "1": 1,
        "burr": 1,
        "burrs": 1,

        "2": 2,
        "flash_burr": 2,
        "flashburr": 2,

        "3": 3,
        "surface_groove_void": 3,
        "surface_groove": 3,
        "groove_void": 3,
        "surface_void": 3,
    }

    if text not in aliases:
        raise ValueError(
            f"Unrecognized frame-level class label: {value!r}"
        )

    return aliases[text]


# =============================================================================
# 6. DATA PREPARATION
# =============================================================================

def load_and_prepare_dataframe(
    csv_path: Path,
) -> pd.DataFrame:
    """
    Load the master dataset, sort within experiment, map labels,
    and recompute first-order differences strictly within each experiment.
    """
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Dataset was not found:\n{csv_path}"
        )

    df = pd.read_csv(csv_path)

    # The master_fusion_dataset_clean.csv used in the earlier H=20 study
    # does not necessarily contain a literal column named "defect_class".
    # Therefore, require only the experiment ID and the 10 raw features here.
    required = (
        ["exp_id"]
        + RAW_FEATURES
    )

    missing = [
        col for col in required
        if col not in df.columns
    ]

    if missing:
        raise KeyError(
            "The dataset is missing required columns:\n"
            + "\n".join(missing)
            + "\n\nAvailable columns are:\n"
            + "\n".join(map(str, df.columns))
        )

    # Frame ordering must be deterministic.
    if "frame_idx" not in df.columns:
        print(
            "\nWARNING: frame_idx was not found. "
            "Creating a within-experiment sequential index."
        )
        df = df.copy()
        df["frame_idx"] = (
            df.groupby("exp_id")
            .cumcount()
            .astype(int)
        )

    df["exp_id"] = df["exp_id"].astype(str)

    df = (
        df.sort_values(
            ["exp_id", "frame_idx"],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    # ---------------------------------------------------------------------
    # FRAME-LEVEL CLASS LABEL
    # ---------------------------------------------------------------------
    # Prefer an explicit class column if one exists. Otherwise reconstruct the
    # same four-class frame label directly from the three class-specific defect
    # area columns used by the H=20 pipeline.
    #
    # Rule when no explicit label column is present:
    #   no defect area > 0        -> Good
    #   otherwise                 -> defect with the largest area
    #
    # A distribution check below protects against accidentally changing the
    # labels relative to the established 2900-row master dataset.
    candidate_label_columns = [
        "defect_class",
        "class_name",
        "class",
        "label",
        "defect_label",
        "weld_quality",
        "quality_class",
    ]

    explicit_label_col = next(
        (
            col for col in candidate_label_columns
            if col in df.columns
        ),
        None,
    )

    if explicit_label_col is not None:
        print(
            f"Using explicit frame-level label column: "
            f"{explicit_label_col}"
        )

        df["target_id"] = (
            df[explicit_label_col]
            .apply(normalize_class_label)
            .astype(int)
        )

    else:
        print(
            "\nNo explicit defect_class column found. "
            "Reconstructing the four-class frame label from "
            "class-specific defect areas."
        )

        area_cols = [
            "Burrs_area_mm2",
            "flash_burr_area_mm2",
            "surface_groove_void_area_mm2",
        ]

        for col in area_cols:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce",
            )

        if df[area_cols].isna().any().any():
            raise ValueError(
                "Cannot reconstruct class labels because NaN/non-numeric "
                "values are present in the defect-area columns."
            )

        area_values = df[area_cols].to_numpy(dtype=float)

        # Start every frame as Good.
        target_id = np.zeros(
            len(df),
            dtype=np.int64,
        )

        has_defect = (
            np.max(
                area_values,
                axis=1,
            )
            > 0.0
        )

        # argmax gives 0/1/2 for Burr/Flash-burr/Groove;
        # +1 maps these to class IDs 1/2/3.
        target_id[has_defect] = (
            np.argmax(
                area_values[has_defect],
                axis=1,
            )
            + 1
        )

        df["target_id"] = target_id

        reconstructed_counts = np.bincount(
            df["target_id"].to_numpy(dtype=int),
            minlength=NUM_CLASSES,
        )

        expected_frame_counts = np.array(
            [537, 1077, 620, 666],
            dtype=int,
        )

        print(
            "Reconstructed frame-level class distribution:"
        )
        for class_id, class_name in enumerate(CLASS_NAMES):
            print(
                f"  {class_name:<22}: "
                f"{reconstructed_counts[class_id]}"
            )

        if not np.array_equal(
            reconstructed_counts,
            expected_frame_counts,
        ):
            raise ValueError(
                "\nThe reconstructed frame-level class distribution does "
                "not match the established H=20 master-dataset distribution.\n"
                f"Expected: {expected_frame_counts.tolist()}\n"
                f"Obtained: {reconstructed_counts.tolist()}\n\n"
                "To avoid silently changing the ablation labels, execution "
                "has been stopped. Please run:\n"
                "    print(pd.read_csv(DATASET_PATH).columns.tolist())\n"
                "and share the output."
            )

        print(
            "Frame-label reconstruction check: PASSED."
        )

    # Convert raw features to numeric and stop if data are invalid.
    for col in RAW_FEATURES:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce",
        )

    if df[RAW_FEATURES].isna().any().any():
        bad_cols = (
            df[RAW_FEATURES]
            .columns[
                df[RAW_FEATURES]
                .isna()
                .any()
            ]
            .tolist()
        )
        raise ValueError(
            "NaN/non-numeric values were found in "
            f"raw feature columns: {bad_cols}"
        )

    # IMPORTANT:
    # Recompute all first-order differences WITHIN EXPERIMENT.
    # This prevents a derivative from ever using the last frame of
    # a different welding experiment.
    for raw_col in RAW_FEATURES:
        diff_col = f"{raw_col}_diff"

        df[diff_col] = (
            df.groupby(
                "exp_id",
                sort=False,
            )[raw_col]
            .diff()
            .fillna(0.0)
            .astype(float)
        )

    return df


# =============================================================================
# 7. TEMPORAL SEQUENCE CONSTRUCTION
# =============================================================================

def build_full_sequences(
    df: pd.DataFrame,
):
    """
    Build the full 20-feature sequence tensor ONCE.

    For each valid sequence:
      input = 20 historical consecutive frames
      target = label 5 frames after the final input frame

    This guarantees that all ablation configurations use exactly the
    same temporal samples and target labels.
    """
    X_list = []
    y_list = []
    meta_rows = []

    sequence_id = 0

    for exp_id, exp_df in df.groupby(
        "exp_id",
        sort=True,
    ):
        exp_df = (
            exp_df.sort_values(
                "frame_idx",
                kind="mergesort",
            )
            .reset_index(drop=True)
        )

        n_rows = len(exp_df)

        n_valid = (
            n_rows
            - SEQUENCE_LENGTH
            - FUTURE_HORIZON
            + 1
        )

        if n_valid <= 0:
            print(
                f"WARNING: {exp_id} has too few rows "
                "for sequence construction."
            )
            continue

        full_values = (
            exp_df[FULL_FEATURES]
            .to_numpy(dtype=np.float32)
        )

        target_values = (
            exp_df["target_id"]
            .to_numpy(dtype=np.int64)
        )

        frame_values = (
            exp_df["frame_idx"]
            .to_numpy()
        )

        for start in range(n_valid):
            end = start + SEQUENCE_LENGTH

            # Last historical input is end - 1.
            # H frames ahead therefore occurs at:
            target_pos = (
                end
                + FUTURE_HORIZON
                - 1
            )

            X_seq = full_values[start:end]
            y_target = target_values[target_pos]

            X_list.append(X_seq)
            y_list.append(y_target)

            meta_rows.append(
                {
                    "sequence_id": sequence_id,
                    "exp_id": exp_id,
                    "input_start_frame": frame_values[start],
                    "input_end_frame": frame_values[end - 1],
                    "target_frame": frame_values[target_pos],
                    "target_id": int(y_target),
                }
            )

            sequence_id += 1

    X_full = np.asarray(
        X_list,
        dtype=np.float32,
    )

    y = np.asarray(
        y_list,
        dtype=np.int64,
    )

    meta_df = pd.DataFrame(meta_rows)

    if len(X_full) != len(y):
        raise RuntimeError(
            "X/y sequence count mismatch."
        )

    if len(meta_df) != len(y):
        raise RuntimeError(
            "Metadata/target sequence count mismatch."
        )

    return X_full, y, meta_df


# =============================================================================
# 8. VALIDATION-PAIR SELECTION
# =============================================================================

def harmonic_mean_proportions(
    class_counts: np.ndarray,
) -> float:
    """
    Harmonic mean of class proportions.

    If any class is absent, return 0.
    This reproduces the class-balance score used in the corrected
    grouped protocol.
    """
    counts = np.asarray(
        class_counts,
        dtype=float,
    )

    total = counts.sum()

    if total <= 0:
        return 0.0

    proportions = counts / total

    if np.any(proportions <= 0):
        return 0.0

    return (
        len(proportions)
        / np.sum(1.0 / proportions)
    )


def rank_validation_pairs(
    meta_df: pd.DataFrame,
    development_experiments: list[str],
) -> pd.DataFrame:
    """
    Rank every two-experiment validation pair using development data only.

    Priority:
      1. Number of represented classes (descending)
      2. Minimum support among the four classes (descending)
      3. Harmonic-mean balance score (descending)
      4. Number of validation sequences (descending)
      5. Experiment names (ascending; deterministic tie break)
    """
    records = []

    dev_exps = sorted(
        development_experiments
    )

    for exp_a, exp_b in itertools.combinations(
        dev_exps,
        2,
    ):
        mask = meta_df["exp_id"].isin(
            [exp_a, exp_b]
        )

        pair_targets = (
            meta_df.loc[
                mask,
                "target_id",
            ]
            .to_numpy(dtype=int)
        )

        counts = np.bincount(
            pair_targets,
            minlength=NUM_CLASSES,
        )

        represented = int(
            np.sum(counts > 0)
        )

        minimum_support = int(
            np.min(counts)
        )

        balance = harmonic_mean_proportions(
            counts
        )

        records.append(
            {
                "exp_a": exp_a,
                "exp_b": exp_b,
                "represented_classes": represented,
                "minimum_class_support": minimum_support,
                "balance_score": balance,
                "validation_samples": int(counts.sum()),
                "Good": int(counts[0]),
                "Burr": int(counts[1]),
                "Flash_burr": int(counts[2]),
                "Surface_groove_void": int(counts[3]),
            }
        )

    ranking = pd.DataFrame(records)

    ranking = ranking.sort_values(
        by=[
            "represented_classes",
            "minimum_class_support",
            "balance_score",
            "validation_samples",
            "exp_a",
            "exp_b",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            True,
            True,
        ],
        kind="mergesort",
    ).reset_index(drop=True)

    return ranking


def build_fold_plan(
    meta_df: pd.DataFrame,
):
    """
    Create one deterministic train/validation/test plan per outer fold.

    The same fold plan is reused by every ablation configuration.
    """
    all_experiments = sorted(
        meta_df["exp_id"]
        .unique()
        .tolist()
    )

    if all_experiments != EXPECTED_INTERNAL_EXPERIMENTS:
        raise ValueError(
            "\nExperiment IDs do not match the corrected "
            "11-experiment grouped protocol.\n"
            f"Expected:\n{EXPECTED_INTERNAL_EXPERIMENTS}\n"
            f"Found:\n{all_experiments}"
        )

    fold_plans = []
    validation_tables = []

    for fold_index, test_exps in enumerate(
        OUTER_TEST_FOLDS,
        start=1,
    ):
        development_exps = sorted(
            set(all_experiments)
            - set(test_exps)
        )

        ranking = rank_validation_pairs(
            meta_df,
            development_exps,
        )

        best_pair = ranking.iloc[0]

        val_exps = sorted(
            [
                str(best_pair["exp_a"]),
                str(best_pair["exp_b"]),
            ]
        )

        train_exps = sorted(
            set(development_exps)
            - set(val_exps)
        )

        # Leakage checks.
        train_set = set(train_exps)
        val_set = set(val_exps)
        test_set = set(test_exps)

        if train_set & val_set:
            raise RuntimeError(
                f"Fold {fold_index}: train/val overlap."
            )

        if train_set & test_set:
            raise RuntimeError(
                f"Fold {fold_index}: train/test overlap."
            )

        if val_set & test_set:
            raise RuntimeError(
                f"Fold {fold_index}: val/test overlap."
            )

        train_idx = np.where(
            meta_df["exp_id"]
            .isin(train_exps)
            .to_numpy()
        )[0]

        val_idx = np.where(
            meta_df["exp_id"]
            .isin(val_exps)
            .to_numpy()
        )[0]

        test_idx = np.where(
            meta_df["exp_id"]
            .isin(test_exps)
            .to_numpy()
        )[0]

        fold_plans.append(
            {
                "fold": fold_index,
                "development_experiments": development_exps,
                "train_experiments": train_exps,
                "validation_experiments": val_exps,
                "test_experiments": list(test_exps),
                "train_idx": train_idx,
                "val_idx": val_idx,
                "test_idx": test_idx,
            }
        )

        table = ranking.copy()
        table.insert(
            0,
            "fold",
            fold_index,
        )
        validation_tables.append(table)

    # Every sequence should appear in exactly one outer test fold.
    all_test_indices = np.concatenate(
        [plan["test_idx"] for plan in fold_plans]
    )

    unique_test_indices = np.unique(
        all_test_indices
    )

    if len(all_test_indices) != len(meta_df):
        raise RuntimeError(
            "Outer folds do not contain exactly one test assignment "
            "for every temporal sequence."
        )

    if len(unique_test_indices) != len(meta_df):
        raise RuntimeError(
            "Some temporal sequences occur in multiple outer test folds."
        )

    validation_rankings = pd.concat(
        validation_tables,
        ignore_index=True,
    )

    return fold_plans, validation_rankings


# =============================================================================
# 9. MODEL
# =============================================================================

class MultiClassFusionLSTM(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int = HIDDEN_SIZE,
        num_layers: int = NUM_LAYERS,
        num_classes: int = NUM_CLASSES,
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=LSTM_DROPOUT,
        )

        self.fc = nn.Sequential(
            nn.Linear(
                hidden_size,
                32,
            ),
            nn.ReLU(),
            nn.Dropout(
                FC_DROPOUT
            ),
            nn.Linear(
                32,
                num_classes,
            ),
        )

    def forward(self, x):
        out, _ = self.lstm(x)

        return self.fc(
            out[:, -1, :]
        )


# =============================================================================
# 10. SCALING AND TENSORS
# =============================================================================

def fit_and_apply_scaler(
    X_train: np.ndarray,
    X_val: np.ndarray,
    X_test: np.ndarray,
):
    """
    Fit StandardScaler using TRAINING SEQUENCES ONLY.

    The time dimension is flattened so one scaler value is learned
    per feature and then applied identically at every timestep.
    """
    n_features = X_train.shape[-1]

    scaler = StandardScaler()

    scaler.fit(
        X_train.reshape(
            -1,
            n_features,
        )
    )

    def transform(X):
        transformed = scaler.transform(
            X.reshape(
                -1,
                n_features,
            )
        )

        return transformed.reshape(
            X.shape
        ).astype(np.float32)

    return (
        transform(X_train),
        transform(X_val),
        transform(X_test),
        scaler,
    )


def calculate_class_weights(
    y_train: np.ndarray,
) -> np.ndarray:
    """
    Balanced inverse-frequency class weights:
        total / (n_classes * class_count)
    """
    counts = np.bincount(
        y_train,
        minlength=NUM_CLASSES,
    ).astype(float)

    if np.any(counts == 0):
        raise RuntimeError(
            "A training subset is missing at least one class. "
            f"Counts: {counts.astype(int)}"
        )

    weights = (
        len(y_train)
        / (
            NUM_CLASSES
            * counts
        )
    )

    return weights.astype(
        np.float32
    )


def make_loader(
    X: np.ndarray,
    y: np.ndarray,
    *,
    batch_size: int,
    shuffle: bool,
    seed: int | None = None,
):
    dataset = TensorDataset(
        torch.from_numpy(
            X.astype(np.float32)
        ),
        torch.from_numpy(
            y.astype(np.int64)
        ),
    )

    generator = None

    if shuffle:
        generator = torch.Generator()

        if seed is None:
            seed = 0

        generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        generator=generator,
    )


# =============================================================================
# 11. TRAINING / EVALUATION
# =============================================================================

def mean_loss(
    model,
    loader,
    criterion,
    device,
) -> float:
    model.eval()

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = criterion(
                logits,
                yb,
            )

            batch_n = yb.size(0)

            total_loss += (
                loss.item()
                * batch_n
            )

            total_samples += batch_n

    return (
        total_loss
        / max(
            total_samples,
            1,
        )
    )


def train_one_model(
    X_train,
    y_train,
    X_val,
    y_val,
    *,
    input_size,
    class_weights,
    fold_index,
    seed,
    configuration_name,
    device,
):
    """
    Train exactly 60 epochs and retain the parameters from the
    LOWEST VALIDATION LOSS.
    """
    set_global_seed(seed)

    model = MultiClassFusionLSTM(
        input_size=input_size,
    ).to(device)

    weight_tensor = torch.tensor(
        class_weights,
        dtype=torch.float32,
        device=device,
    )

    criterion = nn.CrossEntropyLoss(
        weight=weight_tensor,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
    )

    train_loader = make_loader(
        X_train,
        y_train,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=seed,
    )

    val_loader = make_loader(
        X_val,
        y_val,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    best_val_loss = float("inf")
    best_epoch = -1
    best_state = None

    history_rows = []

    for epoch in range(
        1,
        EPOCHS + 1,
    ):
        model.train()

        train_loss_sum = 0.0
        train_samples = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(xb)

            loss = criterion(
                logits,
                yb,
            )

            loss.backward()

            optimizer.step()

            batch_n = yb.size(0)

            train_loss_sum += (
                loss.item()
                * batch_n
            )

            train_samples += batch_n

        train_loss = (
            train_loss_sum
            / max(
                train_samples,
                1,
            )
        )

        val_loss = mean_loss(
            model,
            val_loader,
            criterion,
            device,
        )

        history_rows.append(
            {
                "configuration": configuration_name,
                "fold": fold_index,
                "seed": seed,
                "epoch": epoch,
                "train_loss": train_loss,
                "validation_loss": val_loss,
            }
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(
                model.state_dict()
            )

        if (
            epoch == 1
            or epoch % 10 == 0
            or epoch == EPOCHS
        ):
            print(
                f"{configuration_name} | "
                f"Fold {fold_index} | Seed {seed} | "
                f"Epoch {epoch:02d}/{EPOCHS} | "
                f"Train loss: {train_loss:.6f} | "
                f"Val loss: {val_loss:.6f}"
            )

    if best_state is None:
        raise RuntimeError(
            "No model checkpoint was selected."
        )

    model.load_state_dict(
        best_state
    )

    return (
        model,
        best_epoch,
        best_val_loss,
        history_rows,
    )


def predict(
    model,
    X,
    y,
    *,
    device,
):
    loader = make_loader(
        X,
        y,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)

            logits = model(xb)

            pred = torch.argmax(
                logits,
                dim=1,
            )

            y_true.extend(
                yb.numpy().tolist()
            )

            y_pred.extend(
                pred.cpu()
                .numpy()
                .tolist()
            )

    return (
        np.asarray(
            y_true,
            dtype=int,
        ),
        np.asarray(
            y_pred,
            dtype=int,
        ),
    )


# =============================================================================
# 12. METRICS
# =============================================================================

def compute_metrics(
    y_true,
    y_pred,
):
    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "macro_precision": precision_score(
            y_true,
            y_pred,
            labels=list(
                range(NUM_CLASSES)
            ),
            average="macro",
            zero_division=0,
        ),
        "macro_recall": recall_score(
            y_true,
            y_pred,
            labels=list(
                range(NUM_CLASSES)
            ),
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=list(
                range(NUM_CLASSES)
            ),
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            labels=list(
                range(NUM_CLASSES)
            ),
            average="weighted",
            zero_division=0,
        ),
    }


def summarize_metric_values(
    values,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    n = len(values)

    mean = float(
        np.mean(values)
    )

    sd = float(
        np.std(
            values,
            ddof=1,
        )
    ) if n > 1 else float("nan")

    if n > 1:
        half_width = (
            1.96
            * sd
            / math.sqrt(n)
        )
    else:
        half_width = float("nan")

    return (
        mean,
        sd,
        mean - half_width,
        mean + half_width,
    )


# =============================================================================
# 13. PRINT HELPERS
# =============================================================================

def print_distribution(
    title,
    y_values,
):
    counts = np.bincount(
        y_values,
        minlength=NUM_CLASSES,
    )

    total = counts.sum()

    print(
        f"\n{title}"
    )

    for i, name in enumerate(
        CLASS_NAMES
    ):
        pct = (
            100.0
            * counts[i]
            / total
            if total > 0
            else 0.0
        )

        print(
            f"  {name:<22}: "
            f"{counts[i]:5d} "
            f"({pct:6.2f}%)"
        )


# =============================================================================
# 14. MAIN STUDY
# =============================================================================

def main():
    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print(
        "=" * 130
    )
    print(
        "FINAL 5-FOLD EXPERIMENT-GROUPED × 10-SEED H=10 ABLATION STUDY"
    )
    print(
        "=" * 130
    )

    print(
        f"Device: {device}"
    )
    print(
        f"Dataset: {DATASET_PATH}"
    )
    print(
        f"Output directory: {OUTPUT_DIR}"
    )
    print(
        f"Historical sequence length: {SEQUENCE_LENGTH}"
    )
    print(
        f"Forecast horizon: {FUTURE_HORIZON}"
    )
    print(
        "Approximate forecast time: "
        f"{FUTURE_HORIZON / FPS:.6f} s"
    )
    print(
        f"Seeds: {SEEDS}"
    )
    print(
        "Expected total model runs: "
        f"{len(CONFIGURATIONS) * len(OUTER_TEST_FOLDS) * len(SEEDS)}"
    )

    # -------------------------------------------------------------------------
    # Load and prepare dataframe
    # -------------------------------------------------------------------------
    df = load_and_prepare_dataframe(
        DATASET_PATH
    )

    print(
        f"\nDataset shape: {df.shape}"
    )

    experiments = sorted(
        df["exp_id"]
        .unique()
        .tolist()
    )

    print(
        "\nExperiments:"
    )
    print(
        experiments
    )

    print_distribution(
        "FRAME-LEVEL CLASS DISTRIBUTION",
        df["target_id"]
        .to_numpy(dtype=int),
    )

    # Save prepared frame-level data with within-experiment derivatives.
    df.to_csv(
        OUTPUT_DIR
        / "prepared_frame_level_dataset_H10.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # Build sequences ONCE using all 20 features
    # -------------------------------------------------------------------------
    X_full, y_all, meta_df = build_full_sequences(
        df
    )

    print(
        f"\nTotal temporal sequences: {len(y_all)}"
    )
    print(
        f"Full sequence tensor shape: {X_full.shape}"
    )

    if len(y_all) != EXPECTED_SEQUENCE_COUNT:
        print(
            "\nWARNING: Expected "
            f"{EXPECTED_SEQUENCE_COUNT} sequences for the "
            "2900-row / 11-experiment dataset at H=10, "
            f"but obtained {len(y_all)}."
        )
    else:
        print(
            "Sequence-count check: PASSED "
            f"({EXPECTED_SEQUENCE_COUNT})"
        )

    print_distribution(
        "SEQUENCE-LEVEL TARGET DISTRIBUTION",
        y_all,
    )

    meta_df.to_csv(
        OUTPUT_DIR
        / "sequence_metadata_H10.csv",
        index=False,
    )

    # Per-experiment target distribution.
    per_exp_rows = []

    for exp_id in experiments:
        mask = (
            meta_df["exp_id"]
            == exp_id
        )

        counts = np.bincount(
            y_all[
                mask.to_numpy()
            ],
            minlength=NUM_CLASSES,
        )

        per_exp_rows.append(
            {
                "exp_id": exp_id,
                "n_sequences": int(
                    counts.sum()
                ),
                "Good": int(counts[0]),
                "Burr": int(counts[1]),
                "Flash_burr": int(counts[2]),
                "Surface_groove_void": int(counts[3]),
            }
        )

    per_exp_df = pd.DataFrame(
        per_exp_rows
    )

    print(
        "\nPer-experiment H=10 target distribution:"
    )
    print(
        per_exp_df.to_string(
            index=False
        )
    )

    per_exp_df.to_csv(
        OUTPUT_DIR
        / "per_experiment_target_distribution_H10.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # Fixed outer folds + deterministic development-only validation selection
    # -------------------------------------------------------------------------
    fold_plans, validation_rankings = build_fold_plan(
        meta_df
    )

    validation_rankings.to_csv(
        OUTPUT_DIR
        / "validation_pair_candidates_H10.csv",
        index=False,
    )

    split_rows = []

    for plan in fold_plans:
        fold = plan["fold"]

        print(
            "\n"
            + "=" * 110
        )
        print(
            f"OUTER FOLD {fold}/5"
        )
        print(
            "=" * 110
        )

        print(
            "\nDevelopment experiments:"
        )
        print(
            plan[
                "development_experiments"
            ]
        )

        print(
            "\nHeld-out test experiments:"
        )
        print(
            plan[
                "test_experiments"
            ]
        )

        print(
            "\nSelected validation experiments:"
        )
        print(
            plan[
                "validation_experiments"
            ]
        )

        print(
            "\nTraining experiments:"
        )
        print(
            plan[
                "train_experiments"
            ]
        )

        print(
            "\nSequence counts:"
        )
        print(
            "Training:",
            len(
                plan["train_idx"]
            ),
        )
        print(
            "Validation:",
            len(
                plan["val_idx"]
            ),
        )
        print(
            "Test:",
            len(
                plan["test_idx"]
            ),
        )

        print_distribution(
            "TRAINING target distribution",
            y_all[
                plan["train_idx"]
            ],
        )

        print_distribution(
            "VALIDATION target distribution",
            y_all[
                plan["val_idx"]
            ],
        )

        print_distribution(
            "TEST target distribution",
            y_all[
                plan["test_idx"]
            ],
        )

        split_rows.append(
            {
                "fold": fold,
                "training_experiments": "|".join(
                    plan[
                        "train_experiments"
                    ]
                ),
                "validation_experiments": "|".join(
                    plan[
                        "validation_experiments"
                    ]
                ),
                "test_experiments": "|".join(
                    plan[
                        "test_experiments"
                    ]
                ),
                "training_sequences": len(
                    plan["train_idx"]
                ),
                "validation_sequences": len(
                    plan["val_idx"]
                ),
                "test_sequences": len(
                    plan["test_idx"]
                ),
            }
        )

    pd.DataFrame(
        split_rows
    ).to_csv(
        OUTPUT_DIR
        / "outer_fold_split_plan_H10.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # Feature indices into one shared sequence tensor.
    # -------------------------------------------------------------------------
    feature_to_index = {
        name: idx
        for idx, name
        in enumerate(FULL_FEATURES)
    }

    configuration_indices = {
        config_name: [
            feature_to_index[name]
            for name in feature_names
        ]
        for config_name, feature_names
        in CONFIGURATIONS.items()
    }

    print(
        "\nConfigurations:"
    )

    for name, features in CONFIGURATIONS.items():
        print(
            f"  {name}: {len(features)} features"
        )

    # -------------------------------------------------------------------------
    # Storage
    # -------------------------------------------------------------------------
    fold_metric_rows = []
    prediction_rows = []
    training_history_rows = []
    model_selection_rows = []

    # -------------------------------------------------------------------------
    # Train all 200 models
    # -------------------------------------------------------------------------
    for config_name, feature_names in CONFIGURATIONS.items():
        print(
            "\n\n"
            + "#" * 130
        )
        print(
            config_name
        )
        print(
            f"Number of features: {len(feature_names)}"
        )
        print(
            "#" * 130
        )

        feature_idx = configuration_indices[
            config_name
        ]

        # Every configuration derives from the SAME sequence objects.
        X_config = X_full[
            :,
            :,
            feature_idx,
        ]

        for plan in fold_plans:
            fold = plan["fold"]

            train_idx = plan["train_idx"]
            val_idx = plan["val_idx"]
            test_idx = plan["test_idx"]

            X_train_raw = X_config[
                train_idx
            ]
            X_val_raw = X_config[
                val_idx
            ]
            X_test_raw = X_config[
                test_idx
            ]

            y_train = y_all[
                train_idx
            ]
            y_val = y_all[
                val_idx
            ]
            y_test = y_all[
                test_idx
            ]

            # Scaling is fit ON TRAINING DATA ONLY and reused across seeds
            # for this configuration/fold.
            (
                X_train,
                X_val,
                X_test,
                scaler,
            ) = fit_and_apply_scaler(
                X_train_raw,
                X_val_raw,
                X_test_raw,
            )

            # Class weights are based only on training targets.
            class_weights = calculate_class_weights(
                y_train
            )

            print(
                "\n"
                + "-" * 110
            )
            print(
                f"{config_name} | OUTER FOLD {fold}"
            )
            print(
                "Train experiments:",
                plan[
                    "train_experiments"
                ],
            )
            print(
                "Validation experiments:",
                plan[
                    "validation_experiments"
                ],
            )
            print(
                "Test experiments:",
                plan[
                    "test_experiments"
                ],
            )
            print(
                "Training class counts:",
                np.bincount(
                    y_train,
                    minlength=NUM_CLASSES,
                ),
            )
            print(
                "Training class weights:",
                np.round(
                    class_weights,
                    6,
                ),
            )
            print(
                "-" * 110
            )

            # Optional scaling parameters for auditability.
            scaler_table = pd.DataFrame(
                {
                    "feature": feature_names,
                    "mean": scaler.mean_,
                    "scale": scaler.scale_,
                }
            )

            safe_name = (
                config_name
                .replace(" ", "_")
                .replace("-", "_")
            )

            scaler_table.to_csv(
                OUTPUT_DIR
                / f"scaler_{safe_name}_fold_{fold}.csv",
                index=False,
            )

            for seed in SEEDS:
                print(
                    "\n"
                    + "=" * 110
                )
                print(
                    f"{config_name} | "
                    f"FOLD {fold} | SEED {seed}"
                )
                print(
                    "=" * 110
                )

                (
                    model,
                    best_epoch,
                    best_val_loss,
                    history_rows,
                ) = train_one_model(
                    X_train,
                    y_train,
                    X_val,
                    y_val,
                    input_size=len(
                        feature_names
                    ),
                    class_weights=class_weights,
                    fold_index=fold,
                    seed=seed,
                    configuration_name=config_name,
                    device=device,
                )

                training_history_rows.extend(
                    history_rows
                )

                print(
                    f"\nBest validation epoch: {best_epoch}"
                )
                print(
                    "Best validation loss: "
                    f"{best_val_loss:.6f}"
                )

                # HELD-OUT TEST IS TOUCHED ONLY HERE.
                (
                    y_true_fold,
                    y_pred_fold,
                ) = predict(
                    model,
                    X_test,
                    y_test,
                    device=device,
                )

                metrics = compute_metrics(
                    y_true_fold,
                    y_pred_fold,
                )

                print(
                    "\nHELD-OUT TEST RESULTS"
                )
                print(
                    f"Accuracy: "
                    f"{100 * metrics['accuracy']:.2f}%"
                )
                print(
                    "Macro Precision: "
                    f"{100 * metrics['macro_precision']:.2f}%"
                )
                print(
                    f"Macro Recall: "
                    f"{100 * metrics['macro_recall']:.2f}%"
                )
                print(
                    f"Macro F1: "
                    f"{100 * metrics['macro_f1']:.2f}%"
                )
                print(
                    f"Weighted F1: "
                    f"{100 * metrics['weighted_f1']:.2f}%"
                )

                fold_metric_rows.append(
                    {
                        "configuration": config_name,
                        "number_of_features": len(
                            feature_names
                        ),
                        "fold": fold,
                        "seed": seed,
                        "best_epoch": best_epoch,
                        "best_validation_loss": best_val_loss,
                        **metrics,
                    }
                )

                model_selection_rows.append(
                    {
                        "configuration": config_name,
                        "fold": fold,
                        "seed": seed,
                        "best_epoch": best_epoch,
                        "best_validation_loss": best_val_loss,
                        "training_experiments": "|".join(
                            plan[
                                "train_experiments"
                            ]
                        ),
                        "validation_experiments": "|".join(
                            plan[
                                "validation_experiments"
                            ]
                        ),
                        "test_experiments": "|".join(
                            plan[
                                "test_experiments"
                            ]
                        ),
                    }
                )

                test_meta = (
                    meta_df.iloc[
                        test_idx
                    ]
                    .reset_index(
                        drop=True
                    )
                )

                if len(test_meta) != len(
                    y_true_fold
                ):
                    raise RuntimeError(
                        "Prediction metadata length mismatch."
                    )

                for row_i in range(
                    len(y_true_fold)
                ):
                    prediction_rows.append(
                        {
                            "configuration": config_name,
                            "number_of_features": len(
                                feature_names
                            ),
                            "seed": seed,
                            "fold": fold,
                            "sequence_id": int(
                                test_meta.loc[
                                    row_i,
                                    "sequence_id",
                                ]
                            ),
                            "exp_id": test_meta.loc[
                                row_i,
                                "exp_id",
                            ],
                            "input_start_frame": test_meta.loc[
                                row_i,
                                "input_start_frame",
                            ],
                            "input_end_frame": test_meta.loc[
                                row_i,
                                "input_end_frame",
                            ],
                            "target_frame": test_meta.loc[
                                row_i,
                                "target_frame",
                            ],
                            "true_id": int(
                                y_true_fold[
                                    row_i
                                ]
                            ),
                            "pred_id": int(
                                y_pred_fold[
                                    row_i
                                ]
                            ),
                            "true_class": CLASS_NAMES[
                                int(
                                    y_true_fold[
                                        row_i
                                    ]
                                )
                            ],
                            "pred_class": CLASS_NAMES[
                                int(
                                    y_pred_fold[
                                        row_i
                                    ]
                                )
                            ],
                        }
                    )

                if SAVE_MODEL_CHECKPOINTS:
                    checkpoint_dir = (
                        OUTPUT_DIR
                        / "checkpoints"
                        / safe_name
                    )

                    checkpoint_dir.mkdir(
                        parents=True,
                        exist_ok=True,
                    )

                    torch.save(
                        {
                            "model_state_dict": model.state_dict(),
                            "configuration": config_name,
                            "feature_cols": feature_names,
                            "sequence_length": SEQUENCE_LENGTH,
                            "future_horizon": FUTURE_HORIZON,
                            "fold": fold,
                            "seed": seed,
                            "best_epoch": best_epoch,
                            "best_validation_loss": best_val_loss,
                            "scaler_mean": scaler.mean_,
                            "scaler_scale": scaler.scale_,
                            "class_names": CLASS_NAMES,
                        },
                        checkpoint_dir
                        / (
                            f"fold_{fold}"
                            f"_seed_{seed}.pth"
                        ),
                    )

                # Release GPU memory between models.
                del model

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    # -------------------------------------------------------------------------
    # Save raw study outputs
    # -------------------------------------------------------------------------
    fold_metrics_df = pd.DataFrame(
        fold_metric_rows
    )

    predictions_df = pd.DataFrame(
        prediction_rows
    )

    history_df = pd.DataFrame(
        training_history_rows
    )

    selection_df = pd.DataFrame(
        model_selection_rows
    )

    fold_metrics_df.to_csv(
        OUTPUT_DIR
        / "per_fold_metrics.csv",
        index=False,
    )

    predictions_df.to_csv(
        OUTPUT_DIR
        / "out_of_fold_predictions.csv",
        index=False,
    )

    history_df.to_csv(
        OUTPUT_DIR
        / "training_history.csv",
        index=False,
    )

    selection_df.to_csv(
        OUTPUT_DIR
        / "model_selection_summary.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # Pooled OOF metrics: PRIMARY evaluation.
    # One pooled score per configuration and seed.
    # -------------------------------------------------------------------------
    pooled_rows = []

    aggregate_cm_rows = []

    for config_name, feature_names in CONFIGURATIONS.items():
        for seed in SEEDS:
            subset = (
                predictions_df[
                    (
                        predictions_df[
                            "configuration"
                        ]
                        == config_name
                    )
                    & (
                        predictions_df[
                            "seed"
                        ]
                        == seed
                    )
                ]
                .sort_values(
                    "sequence_id"
                )
                .reset_index(
                    drop=True
                )
            )

            if len(subset) != len(
                meta_df
            ):
                raise RuntimeError(
                    f"{config_name}, seed {seed}: "
                    "pooled OOF prediction count does not equal "
                    "the total number of temporal sequences."
                )

            if subset[
                "sequence_id"
            ].nunique() != len(
                meta_df
            ):
                raise RuntimeError(
                    f"{config_name}, seed {seed}: "
                    "duplicate/missing OOF sequence IDs."
                )

            y_true = subset[
                "true_id"
            ].to_numpy(
                dtype=int
            )

            y_pred = subset[
                "pred_id"
            ].to_numpy(
                dtype=int
            )

            metrics = compute_metrics(
                y_true,
                y_pred,
            )

            pooled_rows.append(
                {
                    "configuration": config_name,
                    "number_of_features": len(
                        feature_names
                    ),
                    "seed": seed,
                    "n_pooled_oof_sequences": len(
                        y_true
                    ),
                    **metrics,
                }
            )

            cm = confusion_matrix(
                y_true,
                y_pred,
                labels=list(
                    range(NUM_CLASSES)
                ),
            )

            for true_id in range(
                NUM_CLASSES
            ):
                for pred_id in range(
                    NUM_CLASSES
                ):
                    aggregate_cm_rows.append(
                        {
                            "configuration": config_name,
                            "seed": seed,
                            "true_id": true_id,
                            "pred_id": pred_id,
                            "true_class": CLASS_NAMES[
                                true_id
                            ],
                            "pred_class": CLASS_NAMES[
                                pred_id
                            ],
                            "count": int(
                                cm[
                                    true_id,
                                    pred_id,
                                ]
                            ),
                        }
                    )

            print(
                "\n"
                + "=" * 110
            )
            print(
                f"POOLED OUT-OF-FOLD PERFORMANCE | "
                f"{config_name} | SEED {seed}"
            )
            print(
                "=" * 110
            )
            print(
                f"Accuracy: "
                f"{100 * metrics['accuracy']:.2f}%"
            )
            print(
                "Macro Precision: "
                f"{100 * metrics['macro_precision']:.2f}%"
            )
            print(
                f"Macro Recall: "
                f"{100 * metrics['macro_recall']:.2f}%"
            )
            print(
                f"Macro F1: "
                f"{100 * metrics['macro_f1']:.2f}%"
            )
            print(
                f"Weighted F1: "
                f"{100 * metrics['weighted_f1']:.2f}%"
            )

    pooled_df = pd.DataFrame(
        pooled_rows
    )

    pooled_df.to_csv(
        OUTPUT_DIR
        / "pooled_by_seed_metrics.csv",
        index=False,
    )

    pd.DataFrame(
        aggregate_cm_rows
    ).to_csv(
        OUTPUT_DIR
        / "confusion_matrices_by_configuration_seed.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # Final summary across 10 seeds
    # -------------------------------------------------------------------------
    final_rows = []

    metric_names = [
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_f1",
    ]

    for config_name, feature_names in CONFIGURATIONS.items():
        block = pooled_df[
            pooled_df[
                "configuration"
            ]
            == config_name
        ]

        row = {
            "configuration": config_name,
            "number_of_features": len(
                feature_names
            ),
            "number_of_seeds": len(
                block
            ),
        }

        for metric in metric_names:
            (
                mean,
                sd,
                low,
                high,
            ) = summarize_metric_values(
                100.0
                * block[
                    metric
                ].to_numpy()
            )

            row[
                f"{metric}_mean"
            ] = mean

            row[
                f"{metric}_sd"
            ] = sd

            row[
                f"{metric}_ci95_lower"
            ] = low

            row[
                f"{metric}_ci95_upper"
            ] = high

        final_rows.append(
            row
        )

    final_summary_df = pd.DataFrame(
        final_rows
    )

    final_summary_df.to_csv(
        OUTPUT_DIR
        / "final_ablation_summary.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # Manuscript-ready multimodal fusion table
    # -------------------------------------------------------------------------
    fusion_names = [
        "Vision-only LSTM",
        "Sensor-only LSTM",
        "Fusion-LSTM with derivatives",
    ]

    fusion_rows = []

    for model_name in fusion_names:
        row = final_summary_df[
            final_summary_df[
                "configuration"
            ]
            == model_name
        ].iloc[0]

        fusion_rows.append(
            {
                "Model": model_name,
                "Features": int(
                    row[
                        "number_of_features"
                    ]
                ),
                "Accuracy (%)": (
                    f"{row['accuracy_mean']:.2f} "
                    f"± {row['accuracy_sd']:.2f}"
                ),
                "Macro Precision (%)": (
                    f"{row['macro_precision_mean']:.2f} "
                    f"± {row['macro_precision_sd']:.2f}"
                ),
                "Macro Recall (%)": (
                    f"{row['macro_recall_mean']:.2f} "
                    f"± {row['macro_recall_sd']:.2f}"
                ),
                "Macro F1 (%)": (
                    f"{row['macro_f1_mean']:.2f} "
                    f"± {row['macro_f1_sd']:.2f}"
                ),
                "Weighted F1 (%)": (
                    f"{row['weighted_f1_mean']:.2f} "
                    f"± {row['weighted_f1_sd']:.2f}"
                ),
                "Accuracy 95% CI": (
                    f"{row['accuracy_ci95_lower']:.2f}"
                    f"–"
                    f"{row['accuracy_ci95_upper']:.2f}"
                ),
                "Macro F1 95% CI": (
                    f"{row['macro_f1_ci95_lower']:.2f}"
                    f"–"
                    f"{row['macro_f1_ci95_upper']:.2f}"
                ),
            }
        )

    manuscript_fusion_df = pd.DataFrame(
        fusion_rows
    )

    manuscript_fusion_df.to_csv(
        OUTPUT_DIR
        / "manuscript_multimodal_fusion_ablation.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # Manuscript-ready derivative table
    # -------------------------------------------------------------------------
    derivative_names = [
        "Fusion-LSTM without derivatives",
        "Fusion-LSTM with derivatives",
    ]

    derivative_rows = []

    for model_name in derivative_names:
        row = final_summary_df[
            final_summary_df[
                "configuration"
            ]
            == model_name
        ].iloc[0]

        derivative_rows.append(
            {
                "Model": model_name,
                "Features": int(
                    row[
                        "number_of_features"
                    ]
                ),
                "Accuracy (%)": (
                    f"{row['accuracy_mean']:.2f} "
                    f"± {row['accuracy_sd']:.2f}"
                ),
                "Macro F1 (%)": (
                    f"{row['macro_f1_mean']:.2f} "
                    f"± {row['macro_f1_sd']:.2f}"
                ),
                "Accuracy 95% CI": (
                    f"{row['accuracy_ci95_lower']:.2f}"
                    f"–"
                    f"{row['accuracy_ci95_upper']:.2f}"
                ),
                "Macro F1 95% CI": (
                    f"{row['macro_f1_ci95_lower']:.2f}"
                    f"–"
                    f"{row['macro_f1_ci95_upper']:.2f}"
                ),
            }
        )

    manuscript_derivative_df = pd.DataFrame(
        derivative_rows
    )

    manuscript_derivative_df.to_csv(
        OUTPUT_DIR
        / "manuscript_derivative_ablation.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # Paired differences using IDENTICAL seed numbers.
    # -------------------------------------------------------------------------
    full_name = (
        "Fusion-LSTM with derivatives"
    )

    comparison_models = [
        "Sensor-only LSTM",
        "Vision-only LSTM",
        "Fusion-LSTM without derivatives",
    ]

    comparison_metrics = [
        "accuracy",
        "macro_f1",
    ]

    paired_rows = []

    paired_seed_rows = []

    full_df = (
        pooled_df[
            pooled_df[
                "configuration"
            ]
            == full_name
        ]
        .set_index(
            "seed"
        )
        .sort_index()
    )

    for other_name in comparison_models:
        other_df = (
            pooled_df[
                pooled_df[
                    "configuration"
                ]
                == other_name
            ]
            .set_index(
                "seed"
            )
            .sort_index()
        )

        if list(
            full_df.index
        ) != list(
            other_df.index
        ):
            raise RuntimeError(
                "Seed pairing failed."
            )

        for metric in comparison_metrics:
            differences_pp = (
                100.0
                * (
                    full_df[
                        metric
                    ]
                    - other_df[
                        metric
                    ]
                )
            )

            for seed, diff_pp in differences_pp.items():
                paired_seed_rows.append(
                    {
                        "comparison": (
                            f"{full_name} minus "
                            f"{other_name}"
                        ),
                        "metric": metric,
                        "seed": int(seed),
                        "difference_percentage_points": float(
                            diff_pp
                        ),
                    }
                )

            (
                mean_diff,
                sd_diff,
                low_diff,
                high_diff,
            ) = summarize_metric_values(
                differences_pp.to_numpy()
            )

            paired_rows.append(
                {
                    "comparison": (
                        f"{full_name} minus "
                        f"{other_name}"
                    ),
                    "metric": metric,
                    "mean_difference_percentage_points": mean_diff,
                    "sd_difference": sd_diff,
                    "ci95_lower": low_diff,
                    "ci95_upper": high_diff,
                }
            )

    paired_df = pd.DataFrame(
        paired_rows
    )

    paired_seed_df = pd.DataFrame(
        paired_seed_rows
    )

    paired_df.to_csv(
        OUTPUT_DIR
        / "paired_performance_differences.csv",
        index=False,
    )

    paired_seed_df.to_csv(
        OUTPUT_DIR
        / "paired_seed_level_differences.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # Plot 1: all four configurations
    # -------------------------------------------------------------------------
    plot_order = list(
        CONFIGURATIONS.keys()
    )

    plot_block = (
        final_summary_df
        .set_index(
            "configuration"
        )
        .loc[
            plot_order
        ]
    )

    x = np.arange(
        len(plot_order)
    )

    width = 0.36

    fig, ax = plt.subplots(
        figsize=(12, 6)
    )

    ax.bar(
        x - width / 2,
        plot_block[
            "accuracy_mean"
        ],
        width,
        yerr=plot_block[
            "accuracy_sd"
        ],
        capsize=4,
        label="Accuracy",
    )

    ax.bar(
        x + width / 2,
        plot_block[
            "macro_f1_mean"
        ],
        width,
        yerr=plot_block[
            "macro_f1_sd"
        ],
        capsize=4,
        label="Macro F1",
    )

    ax.set_ylabel(
        "Performance (%)"
    )
    ax.set_title(
        "Experiment-grouped H=10 ablation "
        "(5 folds × 10 seeds)"
    )
    ax.set_xticks(
        x
    )
    ax.set_xticklabels(
        plot_order,
        rotation=15,
        ha="right",
    )
    ax.legend()
    ax.grid(
        axis="y",
        alpha=0.25,
    )

    fig.tight_layout()

    fig.savefig(
        OUTPUT_DIR
        / "H10_ablation_all_configurations.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)

    # -------------------------------------------------------------------------
    # Plot 2: derivative contribution
    # -------------------------------------------------------------------------
    derivative_plot = (
        final_summary_df
        .set_index(
            "configuration"
        )
        .loc[
            derivative_names
        ]
    )

    x2 = np.arange(
        len(derivative_names)
    )

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    ax.bar(
        x2 - width / 2,
        derivative_plot[
            "accuracy_mean"
        ],
        width,
        yerr=derivative_plot[
            "accuracy_sd"
        ],
        capsize=4,
        label="Accuracy",
    )

    ax.bar(
        x2 + width / 2,
        derivative_plot[
            "macro_f1_mean"
        ],
        width,
        yerr=derivative_plot[
            "macro_f1_sd"
        ],
        capsize=4,
        label="Macro F1",
    )

    ax.set_ylabel(
        "Performance (%)"
    )
    ax.set_title(
        "Effect of first-order derivatives at H=10"
    )
    ax.set_xticks(
        x2
    )
    ax.set_xticklabels(
        [
            "Without derivatives",
            "With derivatives",
        ]
    )
    ax.legend()
    ax.grid(
        axis="y",
        alpha=0.25,
    )

    fig.tight_layout()

    fig.savefig(
        OUTPUT_DIR
        / "H10_derivative_ablation.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)

    # -------------------------------------------------------------------------
    # Save study settings
    # -------------------------------------------------------------------------
    settings = {
        "dataset_path": str(
            DATASET_PATH
        ),
        "output_dir": str(
            OUTPUT_DIR
        ),
        "sequence_length": SEQUENCE_LENGTH,
        "future_horizon": FUTURE_HORIZON,
        "fps": FPS,
        "approximate_horizon_seconds": (
            FUTURE_HORIZON
            / FPS
        ),
        "seeds": SEEDS,
        "outer_test_folds": OUTER_TEST_FOLDS,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "hidden_size": HIDDEN_SIZE,
        "num_layers": NUM_LAYERS,
        "lstm_dropout": LSTM_DROPOUT,
        "fc_dropout": FC_DROPOUT,
        "class_names": CLASS_NAMES,
        "configurations": CONFIGURATIONS,
    }

    with open(
        OUTPUT_DIR
        / "study_settings.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            settings,
            f,
            indent=2,
        )

    # -------------------------------------------------------------------------
    # Final console output
    # -------------------------------------------------------------------------
    print(
        "\n\n"
        + "=" * 130
    )
    print(
        "FINAL EXPERIMENT-GROUPED H=10 ABLATION SUMMARY"
    )
    print(
        "=" * 130
    )

    display_cols = [
        "configuration",
        "number_of_features",
        "accuracy_mean",
        "accuracy_sd",
        "macro_precision_mean",
        "macro_recall_mean",
        "macro_f1_mean",
        "macro_f1_sd",
        "weighted_f1_mean",
    ]

    print(
        final_summary_df[
            display_cols
        ].to_string(
            index=False
        )
    )

    print(
        "\n"
        + "=" * 120
    )
    print(
        "MANUSCRIPT-READY MULTIMODAL FUSION ABLATION"
    )
    print(
        "=" * 120
    )

    print(
        manuscript_fusion_df[
            [
                "Model",
                "Features",
                "Accuracy (%)",
                "Macro Precision (%)",
                "Macro Recall (%)",
                "Macro F1 (%)",
                "Weighted F1 (%)",
            ]
        ].to_string(
            index=False
        )
    )

    print(
        "\n"
        + "=" * 120
    )
    print(
        "MANUSCRIPT-READY DERIVATIVE ABLATION"
    )
    print(
        "=" * 120
    )

    print(
        manuscript_derivative_df[
            [
                "Model",
                "Features",
                "Accuracy (%)",
                "Macro F1 (%)",
                "Accuracy 95% CI",
                "Macro F1 95% CI",
            ]
        ].to_string(
            index=False
        )
    )

    print(
        "\n"
        + "=" * 120
    )
    print(
        "PAIRED PERFORMANCE DIFFERENCES"
    )
    print(
        "=" * 120
    )

    print(
        paired_df.to_string(
            index=False
        )
    )

    print(
        "\n"
        + "=" * 110
    )
    print(
        "EXPERIMENT-GROUPED H=10 ABLATION STUDY COMPLETE"
    )
    print(
        "=" * 110
    )

    print(
        "\nResults saved to:"
    )
    print(
        OUTPUT_DIR
    )

    print(
        "\nConfigurations evaluated:"
    )

    for name, features in CONFIGURATIONS.items():
        print(
            f"  {name}: {len(features)} features"
        )

    print(
        "\nOuter folds: 5"
    )
    print(
        "Seeds per configuration: 10"
    )
    print(
        "Total models trained: 200"
    )

    print(
        "\nMethodological safeguards:"
    )
    safeguards = [
        "H = 10 frames is fixed for all configurations.",
        "Historical input window = 20 frames for all configurations.",
        "Identical temporal sequences and targets are used for every configuration.",
        "Identical outer experiment folds are used for every configuration.",
        "Two complete validation experiments are selected from development data only.",
        "Validation selection is class-coverage aware.",
        "No experiment appears in more than one of train / validation / test within a fold.",
        "Temporal sequences never cross experiment boundaries.",
        "First-order differences are calculated within each experiment.",
        "StandardScaler is fitted only on training sequences.",
        "Class weights are calculated only from training targets.",
        "Weighted cross-entropy is used for class imbalance.",
        "Lowest validation loss selects the final model.",
        "Test experiments are evaluated only after model selection.",
        "Ten identical training seeds are used for every ablation configuration.",
        "Primary comparison uses pooled out-of-fold predictions across all experiments.",
    ]

    for i, text in enumerate(
        safeguards,
        start=1,
    ):
        print(
            f"{i}. {text}"
        )

    print(
        "\nDone."
    )


if __name__ == "__main__":
    main()


In [ ]:
# -*- coding: utf-8 -*-
"""
FINAL H=10 FUSION-LSTM WITH DERIVATIVES
POOLED OOF CONFUSION MATRIX + PER-CLASS PERFORMANCE
===================================================

IMPORTANT
---------
This script DOES NOT retrain the LSTM.

It reads the out-of-fold predictions produced by the successful:

    H=10
    5-fold experiment-grouped
    10-seed
    ablation study

and analyses ONLY:

    Fusion-LSTM with derivatives

Outputs
-------
1. Per-seed overall metrics
2. Per-seed per-class metrics
3. Mean ± SD per-class performance across 10 seeds
4. Confusion matrix for every seed
5. Row-normalized confusion matrix for every seed
6. Mean confusion matrix across seeds
7. Mean row-normalized confusion matrix across seeds
8. Majority-vote consensus OOF confusion matrix
9. Majority-vote per-class performance
10. Manuscript-ready CSV tables
11. Publication-ready PNG + SVG figures
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)


# =============================================================================
# 1. PATHS
# =============================================================================

PROJECT_DIR = Path.cwd()

ABLATION_DIR = (
    PROJECT_DIR
    / "outputs"
    / "Final_Grouped_H10_Ablation_HorizonFolds_5Fold_10Seeds"
)

PREDICTION_FILE = (
    ABLATION_DIR
    / "out_of_fold_predictions.csv"
)

OUTPUT_DIR = (
    ABLATION_DIR
    / "Final_H10_Fusion_PerClass_Analysis"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 2. FINAL MODEL / STUDY SETTINGS
# =============================================================================

MODEL_NAME = "Fusion-LSTM with derivatives"

SEEDS = list(
    range(42, 52)
)

NUM_CLASSES = 4

CLASS_IDS = [
    0,
    1,
    2,
    3,
]

CLASS_NAMES = [
    "Good",
    "Burr",
    "Flash-burr",
    "Surface-groove/void",
]

EXPECTED_SEQUENCES_PER_SEED = 2581


# =============================================================================
# 3. HELPER FUNCTIONS
# =============================================================================

def safe_divide(
    numerator,
    denominator,
):
    """
    Safe element-wise division.
    """

    numerator = np.asarray(
        numerator,
        dtype=float,
    )

    denominator = np.asarray(
        denominator,
        dtype=float,
    )

    result = np.zeros_like(
        numerator,
        dtype=float,
    )

    np.divide(
        numerator,
        denominator,
        out=result,
        where=denominator != 0,
    )

    return result


def summarize_values(
    values,
):
    """
    Return mean, sample SD and approximate 95% CI.
    """

    values = np.asarray(
        values,
        dtype=float,
    )

    n = len(values)

    mean = float(
        np.mean(values)
    )

    if n > 1:

        sd = float(
            np.std(
                values,
                ddof=1,
            )
        )

        half_width = (
            1.96
            * sd
            / np.sqrt(n)
        )

        ci_lower = (
            mean
            - half_width
        )

        ci_upper = (
            mean
            + half_width
        )

    else:

        sd = float("nan")
        ci_lower = float("nan")
        ci_upper = float("nan")

    return (
        mean,
        sd,
        ci_lower,
        ci_upper,
    )


def calculate_specificity_from_cm(
    cm,
):
    """
    Calculate one-vs-rest specificity for every class.
    """

    total = cm.sum()

    specificity = []

    for class_id in range(
        NUM_CLASSES
    ):

        tp = cm[
            class_id,
            class_id,
        ]

        fn = (
            cm[
                class_id,
                :
            ].sum()
            - tp
        )

        fp = (
            cm[
                :,
                class_id
            ].sum()
            - tp
        )

        tn = (
            total
            - tp
            - fn
            - fp
        )

        denominator = (
            tn
            + fp
        )

        if denominator > 0:

            value = (
                tn
                / denominator
            )

        else:

            value = 0.0

        specificity.append(
            value
        )

    return np.asarray(
        specificity,
        dtype=float,
    )


def calculate_per_class_metrics(
    y_true,
    y_pred,
):
    """
    Precision, recall, F1, support and specificity
    for all four classes.
    """

    (
        precision,
        recall,
        f1,
        support,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=CLASS_IDS,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=CLASS_IDS,
    )

    specificity = (
        calculate_specificity_from_cm(
            cm
        )
    )

    rows = []

    for class_id in CLASS_IDS:

        rows.append(
            {
                "class_id": class_id,

                "class_name":
                    CLASS_NAMES[
                        class_id
                    ],

                "precision":
                    float(
                        precision[
                            class_id
                        ]
                    ),

                "recall":
                    float(
                        recall[
                            class_id
                        ]
                    ),

                "specificity":
                    float(
                        specificity[
                            class_id
                        ]
                    ),

                "f1":
                    float(
                        f1[
                            class_id
                        ]
                    ),

                "support":
                    int(
                        support[
                            class_id
                        ]
                    ),
            }
        )

    return (
        pd.DataFrame(rows),
        cm,
    )


def row_normalize_confusion_matrix(
    cm,
):
    """
    Normalize each true-class row to proportions.
    """

    cm = np.asarray(
        cm,
        dtype=float,
    )

    row_sums = (
        cm.sum(
            axis=1,
            keepdims=True,
        )
    )

    normalized = np.divide(
        cm,
        row_sums,
        out=np.zeros_like(
            cm,
            dtype=float,
        ),
        where=row_sums != 0,
    )

    return normalized


def save_matrix_csv(
    matrix,
    filename,
):
    """
    Save a matrix with class labels.
    """

    df = pd.DataFrame(
        matrix,
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    )

    df.index.name = (
        "True class"
    )

    df.to_csv(
        OUTPUT_DIR
        / filename
    )


def plot_confusion_matrix(
    matrix,
    title,
    filename_base,
    *,
    normalized=False,
):
    """
    Save one publication-ready confusion matrix
    as PNG and SVG.
    """

    matrix = np.asarray(
        matrix
    )

    fig, ax = plt.subplots(
        figsize=(8.5, 7.0)
    )

    image = ax.imshow(
        matrix,
        aspect="auto",
    )

    fig.colorbar(
        image,
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )

    ax.set_xticks(
        np.arange(
            NUM_CLASSES
        )
    )

    ax.set_yticks(
        np.arange(
            NUM_CLASSES
        )
    )

    ax.set_xticklabels(
        CLASS_NAMES,
        rotation=35,
        ha="right",
    )

    ax.set_yticklabels(
        CLASS_NAMES
    )

    ax.set_xlabel(
        "Predicted class"
    )

    ax.set_ylabel(
        "True class"
    )

    ax.set_title(
        title
    )

    threshold = (
        (
            matrix.max()
            + matrix.min()
        )
        / 2.0
        if matrix.size > 0
        else 0
    )

    for i in range(
        NUM_CLASSES
    ):

        for j in range(
            NUM_CLASSES
        ):

            if normalized:

                text = (
                    f"{100 * matrix[i, j]:.1f}%"
                )

            else:

                if np.issubdtype(
                    matrix.dtype,
                    np.integer,
                ):

                    text = (
                        f"{int(matrix[i, j])}"
                    )

                else:

                    text = (
                        f"{matrix[i, j]:.1f}"
                    )

            text_color = (
                "white"
                if matrix[i, j] > threshold
                else "black"
            )

            ax.text(
                j,
                i,
                text,
                ha="center",
                va="center",
                color=text_color,
                fontsize=11,
            )

    fig.tight_layout()

    fig.savefig(
        OUTPUT_DIR
        / f"{filename_base}.png",
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        OUTPUT_DIR
        / f"{filename_base}.svg",
        bbox_inches="tight",
    )

    plt.close(
        fig
    )


# =============================================================================
# 4. LOAD SUCCESSFUL H=10 OOF PREDICTIONS
# =============================================================================

print(
    "=" * 120
)

print(
    "FINAL H=10 FUSION-LSTM WITH DERIVATIVES"
)

print(
    "POOLED OOF CONFUSION MATRIX + PER-CLASS PERFORMANCE"
)

print(
    "=" * 120
)


if not PREDICTION_FILE.exists():

    raise FileNotFoundError(
        "\nCould not find:\n"
        f"{PREDICTION_FILE}\n\n"
        "Run the successful H=10 ablation first."
    )


print(
    "\nReading:"
)

print(
    PREDICTION_FILE
)


predictions = pd.read_csv(
    PREDICTION_FILE
)


# =============================================================================
# 5. VALIDATE INPUT FILE
# =============================================================================

required_columns = [
    "configuration",
    "seed",
    "fold",
    "sequence_id",
    "exp_id",
    "true_id",
    "pred_id",
    "true_class",
    "pred_class",
]


missing_columns = [
    column
    for column in required_columns
    if column not in predictions.columns
]


if missing_columns:

    raise KeyError(
        "\nOOF prediction file is missing columns:\n"
        + "\n".join(
            missing_columns
        )
    )


print(
    f"\nTotal rows in OOF file: "
    f"{len(predictions):,}"
)


print(
    "\nConfigurations available:"
)

for name in sorted(
    predictions[
        "configuration"
    ].unique()
):

    print(
        f"  {name}"
    )


# =============================================================================
# 6. KEEP ONLY FINAL MODEL
# =============================================================================

fusion = (
    predictions[
        predictions[
            "configuration"
        ]
        == MODEL_NAME
    ]
    .copy()
)


if fusion.empty:

    raise RuntimeError(
        "\nCould not find configuration:\n"
        f"{MODEL_NAME}"
    )


fusion[
    "seed"
] = pd.to_numeric(
    fusion[
        "seed"
    ],
    errors="raise",
).astype(int)


fusion[
    "sequence_id"
] = pd.to_numeric(
    fusion[
        "sequence_id"
    ],
    errors="raise",
).astype(int)


fusion[
    "true_id"
] = pd.to_numeric(
    fusion[
        "true_id"
    ],
    errors="raise",
).astype(int)


fusion[
    "pred_id"
] = pd.to_numeric(
    fusion[
        "pred_id"
    ],
    errors="raise",
).astype(int)


available_seeds = sorted(
    fusion[
        "seed"
    ].unique().tolist()
)


if available_seeds != SEEDS:

    raise RuntimeError(
        "\nSeed mismatch.\n"
        f"Expected: {SEEDS}\n"
        f"Found:    {available_seeds}"
    )


print(
    "\nSelected model:"
)

print(
    MODEL_NAME
)


print(
    "\nSeeds:"
)

print(
    available_seeds
)


# =============================================================================
# 7. CRITICAL OOF INTEGRITY CHECKS
# =============================================================================

reference_sequence_ids = None

reference_true_labels = None


for seed in SEEDS:

    seed_df = (
        fusion[
            fusion[
                "seed"
            ]
            == seed
        ]
        .sort_values(
            "sequence_id"
        )
        .reset_index(
            drop=True
        )
    )


    n_rows = len(
        seed_df
    )


    n_unique_sequences = (
        seed_df[
            "sequence_id"
        ].nunique()
    )


    if n_rows != EXPECTED_SEQUENCES_PER_SEED:

        raise RuntimeError(
            f"\nSeed {seed}: expected "
            f"{EXPECTED_SEQUENCES_PER_SEED} OOF predictions "
            f"but found {n_rows}."
        )


    if (
        n_unique_sequences
        != EXPECTED_SEQUENCES_PER_SEED
    ):

        raise RuntimeError(
            f"\nSeed {seed}: duplicate or missing "
            "OOF sequence IDs."
        )


    sequence_ids = (
        seed_df[
            "sequence_id"
        ].to_numpy()
    )


    true_labels = (
        seed_df[
            "true_id"
        ].to_numpy()
    )


    if reference_sequence_ids is None:

        reference_sequence_ids = (
            sequence_ids.copy()
        )

        reference_true_labels = (
            true_labels.copy()
        )

    else:

        if not np.array_equal(
            sequence_ids,
            reference_sequence_ids,
        ):

            raise RuntimeError(
                f"\nSeed {seed}: sequence IDs do not "
                "match the reference seed."
            )


        if not np.array_equal(
            true_labels,
            reference_true_labels,
        ):

            raise RuntimeError(
                f"\nSeed {seed}: true labels do not "
                "match the reference seed."
            )


print(
    "\nOOF integrity check: PASSED"
)

print(
    "Each seed contains exactly "
    f"{EXPECTED_SEQUENCES_PER_SEED} unique "
    "held-out predictions."
)


# =============================================================================
# 8. TRUE CLASS DISTRIBUTION
# =============================================================================

true_distribution = np.bincount(
    reference_true_labels,
    minlength=NUM_CLASSES,
)


true_distribution_df = pd.DataFrame(
    {
        "class_id": CLASS_IDS,
        "class_name": CLASS_NAMES,
        "support": true_distribution,
        "percentage": (
            100.0
            * true_distribution
            / true_distribution.sum()
        ),
    }
)


true_distribution_df.to_csv(
    OUTPUT_DIR
    / "H10_true_class_distribution.csv",
    index=False,
)


print(
    "\n"
    + "=" * 100
)

print(
    "TRUE H=10 OOF CLASS DISTRIBUTION"
)

print(
    "=" * 100
)

print(
    true_distribution_df.to_string(
        index=False
    )
)


# =============================================================================
# 9. PER-SEED ANALYSIS
# =============================================================================

overall_seed_rows = []

per_class_seed_rows = []

confusion_matrices = []

normalized_confusion_matrices = []


for seed in SEEDS:

    seed_df = (
        fusion[
            fusion[
                "seed"
            ]
            == seed
        ]
        .sort_values(
            "sequence_id"
        )
        .reset_index(
            drop=True
        )
    )


    y_true = (
        seed_df[
            "true_id"
        ].to_numpy(
            dtype=int
        )
    )


    y_pred = (
        seed_df[
            "pred_id"
        ].to_numpy(
            dtype=int
        )
    )


    # -------------------------------------------------------------------------
    # Overall metrics
    # -------------------------------------------------------------------------

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )


    macro_precision = precision_score(
        y_true,
        y_pred,
        labels=CLASS_IDS,
        average="macro",
        zero_division=0,
    )


    macro_recall = recall_score(
        y_true,
        y_pred,
        labels=CLASS_IDS,
        average="macro",
        zero_division=0,
    )


    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=CLASS_IDS,
        average="macro",
        zero_division=0,
    )


    weighted_f1 = f1_score(
        y_true,
        y_pred,
        labels=CLASS_IDS,
        average="weighted",
        zero_division=0,
    )


    overall_seed_rows.append(
        {
            "seed": seed,

            "accuracy":
                accuracy,

            "macro_precision":
                macro_precision,

            "macro_recall":
                macro_recall,

            "macro_f1":
                macro_f1,

            "weighted_f1":
                weighted_f1,
        }
    )


    # -------------------------------------------------------------------------
    # Per-class metrics
    # -------------------------------------------------------------------------

    (
        per_class_df,
        cm,
    ) = calculate_per_class_metrics(
        y_true,
        y_pred,
    )


    per_class_df.insert(
        0,
        "seed",
        seed,
    )


    per_class_seed_rows.extend(
        per_class_df.to_dict(
            orient="records"
        )
    )


    # -------------------------------------------------------------------------
    # Confusion matrices
    # -------------------------------------------------------------------------

    cm_normalized = (
        row_normalize_confusion_matrix(
            cm
        )
    )


    confusion_matrices.append(
        cm.astype(float)
    )


    normalized_confusion_matrices.append(
        cm_normalized
    )


    save_matrix_csv(
        cm,
        f"H10_confusion_matrix_seed_{seed}.csv",
    )


    save_matrix_csv(
        cm_normalized,
        (
            "H10_confusion_matrix_normalized_"
            f"seed_{seed}.csv"
        ),
    )


    plot_confusion_matrix(
        cm,
        (
            "H=10 Fusion-LSTM with derivatives\n"
            f"Pooled OOF confusion matrix — Seed {seed}"
        ),
        f"H10_confusion_matrix_seed_{seed}",
        normalized=False,
    )


    plot_confusion_matrix(
        cm_normalized,
        (
            "H=10 Fusion-LSTM with derivatives\n"
            "Row-normalized pooled OOF confusion matrix "
            f"— Seed {seed}"
        ),
        (
            "H10_confusion_matrix_normalized_"
            f"seed_{seed}"
        ),
        normalized=True,
    )


    print(
        "\n"
        + "=" * 100
    )

    print(
        f"SEED {seed}"
    )

    print(
        "=" * 100
    )

    print(
        f"Accuracy        : "
        f"{100 * accuracy:.2f}%"
    )

    print(
        f"Macro Precision : "
        f"{100 * macro_precision:.2f}%"
    )

    print(
        f"Macro Recall    : "
        f"{100 * macro_recall:.2f}%"
    )

    print(
        f"Macro F1        : "
        f"{100 * macro_f1:.2f}%"
    )

    print(
        f"Weighted F1     : "
        f"{100 * weighted_f1:.2f}%"
    )

    print(
        "\nPer-class performance:"
    )

    display = (
        per_class_df[
            [
                "class_name",
                "precision",
                "recall",
                "specificity",
                "f1",
                "support",
            ]
        ]
        .copy()
    )


    for column in [
        "precision",
        "recall",
        "specificity",
        "f1",
    ]:

        display[
            column
        ] = (
            100.0
            * display[
                column
            ]
        )


    print(
        display.to_string(
            index=False,
            float_format=lambda x: f"{x:.2f}",
        )
    )


# =============================================================================
# 10. SAVE PER-SEED RESULTS
# =============================================================================

overall_seed_df = pd.DataFrame(
    overall_seed_rows
)


per_class_seed_df = pd.DataFrame(
    per_class_seed_rows
)


overall_seed_df.to_csv(
    OUTPUT_DIR
    / "H10_overall_metrics_by_seed.csv",
    index=False,
)


per_class_seed_df.to_csv(
    OUTPUT_DIR
    / "H10_per_class_metrics_by_seed.csv",
    index=False,
)


# =============================================================================
# 11. OVERALL PERFORMANCE SUMMARY ACROSS SEEDS
# =============================================================================

overall_summary_rows = []


for metric in [
    "accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
]:

    values = (
        100.0
        * overall_seed_df[
            metric
        ].to_numpy()
    )


    (
        mean,
        sd,
        ci_lower,
        ci_upper,
    ) = summarize_values(
        values
    )


    overall_summary_rows.append(
        {
            "metric": metric,

            "mean_percent":
                mean,

            "sd_percent":
                sd,

            "ci95_lower_percent":
                ci_lower,

            "ci95_upper_percent":
                ci_upper,

            "mean_plus_minus_sd":
                f"{mean:.2f} ± {sd:.2f}",
        }
    )


overall_summary_df = pd.DataFrame(
    overall_summary_rows
)


overall_summary_df.to_csv(
    OUTPUT_DIR
    / "H10_overall_performance_summary.csv",
    index=False,
)


# =============================================================================
# 12. PER-CLASS SUMMARY ACROSS 10 SEEDS
# =============================================================================

per_class_summary_rows = []


for class_id, class_name in zip(
    CLASS_IDS,
    CLASS_NAMES,
):

    class_block = (
        per_class_seed_df[
            per_class_seed_df[
                "class_id"
            ]
            == class_id
        ]
        .copy()
    )


    row = {
        "class_id": class_id,
        "class_name": class_name,
        "support": int(
            true_distribution[
                class_id
            ]
        ),
    }


    for metric in [
        "precision",
        "recall",
        "specificity",
        "f1",
    ]:

        values = (
            100.0
            * class_block[
                metric
            ].to_numpy()
        )


        (
            mean,
            sd,
            ci_lower,
            ci_upper,
        ) = summarize_values(
            values
        )


        row[
            f"{metric}_mean_percent"
        ] = mean


        row[
            f"{metric}_sd_percent"
        ] = sd


        row[
            f"{metric}_ci95_lower_percent"
        ] = ci_lower


        row[
            f"{metric}_ci95_upper_percent"
        ] = ci_upper


        row[
            f"{metric}_mean_plus_minus_sd"
        ] = (
            f"{mean:.2f} ± {sd:.2f}"
        )


    per_class_summary_rows.append(
        row
    )


per_class_summary_df = pd.DataFrame(
    per_class_summary_rows
)


per_class_summary_df.to_csv(
    OUTPUT_DIR
    / "H10_per_class_summary_10seeds.csv",
    index=False,
)


# =============================================================================
# 13. MANUSCRIPT-READY PER-CLASS TABLE
# =============================================================================

manuscript_rows = []


for _, row in (
    per_class_summary_df.iterrows()
):

    manuscript_rows.append(
        {
            "Class":
                row[
                    "class_name"
                ],

            "Support":
                int(
                    row[
                        "support"
                    ]
                ),

            "Precision (%)":
                row[
                    "precision_mean_plus_minus_sd"
                ],

            "Recall (%)":
                row[
                    "recall_mean_plus_minus_sd"
                ],

            "Specificity (%)":
                row[
                    "specificity_mean_plus_minus_sd"
                ],

            "F1-score (%)":
                row[
                    "f1_mean_plus_minus_sd"
                ],
        }
    )


manuscript_df = pd.DataFrame(
    manuscript_rows
)


manuscript_df.to_csv(
    OUTPUT_DIR
    / "H10_manuscript_per_class_performance.csv",
    index=False,
)


# =============================================================================
# 14. MEAN CONFUSION MATRIX ACROSS 10 SEEDS
# =============================================================================

confusion_stack = np.stack(
    confusion_matrices,
    axis=0,
)


normalized_stack = np.stack(
    normalized_confusion_matrices,
    axis=0,
)


mean_cm = np.mean(
    confusion_stack,
    axis=0,
)


sd_cm = np.std(
    confusion_stack,
    axis=0,
    ddof=1,
)


mean_normalized_cm = np.mean(
    normalized_stack,
    axis=0,
)


sd_normalized_cm = np.std(
    normalized_stack,
    axis=0,
    ddof=1,
)


save_matrix_csv(
    mean_cm,
    "H10_mean_confusion_matrix_10seeds.csv",
)


save_matrix_csv(
    sd_cm,
    "H10_sd_confusion_matrix_10seeds.csv",
)


save_matrix_csv(
    mean_normalized_cm,
    (
        "H10_mean_normalized_confusion_"
        "matrix_10seeds.csv"
    ),
)


save_matrix_csv(
    sd_normalized_cm,
    (
        "H10_sd_normalized_confusion_"
        "matrix_10seeds.csv"
    ),
)


plot_confusion_matrix(
    mean_cm,
    (
        "H=10 Fusion-LSTM with derivatives\n"
        "Mean pooled OOF confusion matrix across 10 seeds"
    ),
    "H10_mean_confusion_matrix_10seeds",
    normalized=False,
)


plot_confusion_matrix(
    mean_normalized_cm,
    (
        "H=10 Fusion-LSTM with derivatives\n"
        "Mean row-normalized pooled OOF confusion matrix "
        "across 10 seeds"
    ),
    "H10_mean_normalized_confusion_matrix_10seeds",
    normalized=True,
)


# =============================================================================
# 15. CONFUSION MATRIX CELL SUMMARY
# =============================================================================

cm_cell_rows = []


for true_id in CLASS_IDS:

    for pred_id in CLASS_IDS:

        count_values = (
            confusion_stack[
                :,
                true_id,
                pred_id,
            ]
        )


        normalized_values = (
            100.0
            * normalized_stack[
                :,
                true_id,
                pred_id,
            ]
        )


        (
            count_mean,
            count_sd,
            _,
            _,
        ) = summarize_values(
            count_values
        )


        (
            pct_mean,
            pct_sd,
            pct_low,
            pct_high,
        ) = summarize_values(
            normalized_values
        )


        cm_cell_rows.append(
            {
                "true_id":
                    true_id,

                "true_class":
                    CLASS_NAMES[
                        true_id
                    ],

                "pred_id":
                    pred_id,

                "pred_class":
                    CLASS_NAMES[
                        pred_id
                    ],

                "count_mean":
                    count_mean,

                "count_sd":
                    count_sd,

                "row_percent_mean":
                    pct_mean,

                "row_percent_sd":
                    pct_sd,

                "row_percent_ci95_lower":
                    pct_low,

                "row_percent_ci95_upper":
                    pct_high,
            }
        )


cm_cell_summary_df = pd.DataFrame(
    cm_cell_rows
)


cm_cell_summary_df.to_csv(
    OUTPUT_DIR
    / "H10_confusion_matrix_cell_summary.csv",
    index=False,
)


# =============================================================================
# 16. MAJORITY-VOTE CONSENSUS ACROSS 10 SEEDS
# =============================================================================

pivot = (
    fusion.pivot(
        index="sequence_id",
        columns="seed",
        values="pred_id",
    )
    .sort_index()
)


if list(
    pivot.columns
) != SEEDS:

    raise RuntimeError(
        "Majority-vote seed columns do not match "
        "the expected seeds."
    )


consensus_predictions = []


for sequence_id, row in (
    pivot.iterrows()
):

    votes = (
        row.to_numpy(
            dtype=int
        )
    )


    counts = np.bincount(
        votes,
        minlength=NUM_CLASSES,
    )


    # Deterministic tie handling:
    # np.argmax returns the lowest class ID
    # when two classes receive the same maximum votes.
    consensus_class = int(
        np.argmax(
            counts
        )
    )


    consensus_predictions.append(
        consensus_class
    )


consensus_predictions = np.asarray(
    consensus_predictions,
    dtype=int,
)


# Recover true labels in the same sequence order.
reference_seed_df = (
    fusion[
        fusion[
            "seed"
        ]
        == SEEDS[0]
    ]
    .set_index(
        "sequence_id"
    )
    .loc[
        pivot.index
    ]
)


consensus_true = (
    reference_seed_df[
        "true_id"
    ].to_numpy(
        dtype=int
    )
)


(
    consensus_per_class_df,
    consensus_cm,
) = calculate_per_class_metrics(
    consensus_true,
    consensus_predictions,
)


consensus_normalized_cm = (
    row_normalize_confusion_matrix(
        consensus_cm
    )
)


consensus_accuracy = accuracy_score(
    consensus_true,
    consensus_predictions,
)


consensus_macro_precision = precision_score(
    consensus_true,
    consensus_predictions,
    labels=CLASS_IDS,
    average="macro",
    zero_division=0,
)


consensus_macro_recall = recall_score(
    consensus_true,
    consensus_predictions,
    labels=CLASS_IDS,
    average="macro",
    zero_division=0,
)


consensus_macro_f1 = f1_score(
    consensus_true,
    consensus_predictions,
    labels=CLASS_IDS,
    average="macro",
    zero_division=0,
)


consensus_weighted_f1 = f1_score(
    consensus_true,
    consensus_predictions,
    labels=CLASS_IDS,
    average="weighted",
    zero_division=0,
)


consensus_overall_df = pd.DataFrame(
    [
        {
            "accuracy":
                consensus_accuracy,

            "macro_precision":
                consensus_macro_precision,

            "macro_recall":
                consensus_macro_recall,

            "macro_f1":
                consensus_macro_f1,

            "weighted_f1":
                consensus_weighted_f1,
        }
    ]
)


consensus_overall_df.to_csv(
    OUTPUT_DIR
    / "H10_consensus_overall_metrics.csv",
    index=False,
)


consensus_per_class_df.to_csv(
    OUTPUT_DIR
    / "H10_consensus_per_class_metrics.csv",
    index=False,
)


save_matrix_csv(
    consensus_cm,
    "H10_consensus_confusion_matrix.csv",
)


save_matrix_csv(
    consensus_normalized_cm,
    (
        "H10_consensus_normalized_"
        "confusion_matrix.csv"
    ),
)


plot_confusion_matrix(
    consensus_cm,
    (
        "H=10 Fusion-LSTM with derivatives\n"
        "10-seed majority-vote OOF confusion matrix"
    ),
    "H10_consensus_confusion_matrix",
    normalized=False,
)


plot_confusion_matrix(
    consensus_normalized_cm,
    (
        "H=10 Fusion-LSTM with derivatives\n"
        "10-seed majority-vote row-normalized "
        "OOF confusion matrix"
    ),
    "H10_consensus_normalized_confusion_matrix",
    normalized=True,
)


# =============================================================================
# 17. SAVE CONSENSUS PREDICTIONS
# =============================================================================

consensus_prediction_df = pd.DataFrame(
    {
        "sequence_id":
            pivot.index.to_numpy(),

        "true_id":
            consensus_true,

        "true_class":
            [
                CLASS_NAMES[
                    value
                ]
                for value
                in consensus_true
            ],

        "consensus_pred_id":
            consensus_predictions,

        "consensus_pred_class":
            [
                CLASS_NAMES[
                    value
                ]
                for value
                in consensus_predictions
            ],
    }
)


consensus_prediction_df.to_csv(
    OUTPUT_DIR
    / "H10_consensus_predictions.csv",
    index=False,
)


# =============================================================================
# 18. PRINT FINAL RESULTS
# =============================================================================

print(
    "\n\n"
    + "=" * 120
)

print(
    "FINAL H=10 OVERALL PERFORMANCE ACROSS 10 SEEDS"
)

print(
    "=" * 120
)


print(
    overall_summary_df[
        [
            "metric",
            "mean_plus_minus_sd",
            "ci95_lower_percent",
            "ci95_upper_percent",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n"
    + "=" * 120
)

print(
    "FINAL H=10 PER-CLASS PERFORMANCE"
)

print(
    "=" * 120
)


print(
    manuscript_df.to_string(
        index=False
    )
)


print(
    "\n"
    + "=" * 120
)

print(
    "MEAN ROW-NORMALIZED CONFUSION MATRIX ACROSS 10 SEEDS (%)"
)

print(
    "=" * 120
)


mean_normalized_display = pd.DataFrame(
    100.0
    * mean_normalized_cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
)


print(
    mean_normalized_display.to_string(
        float_format=lambda x: f"{x:.2f}"
    )
)


print(
    "\n"
    + "=" * 120
)

print(
    "10-SEED MAJORITY-VOTE CONSENSUS PERFORMANCE"
)

print(
    "=" * 120
)


print(
    f"Accuracy        : "
    f"{100 * consensus_accuracy:.2f}%"
)

print(
    f"Macro Precision : "
    f"{100 * consensus_macro_precision:.2f}%"
)

print(
    f"Macro Recall    : "
    f"{100 * consensus_macro_recall:.2f}%"
)

print(
    f"Macro F1        : "
    f"{100 * consensus_macro_f1:.2f}%"
)

print(
    f"Weighted F1     : "
    f"{100 * consensus_weighted_f1:.2f}%"
)


print(
    "\nConsensus per-class performance:"
)


consensus_display = (
    consensus_per_class_df.copy()
)


for column in [
    "precision",
    "recall",
    "specificity",
    "f1",
]:

    consensus_display[
        column
    ] = (
        100.0
        * consensus_display[
            column
        ]
    )


print(
    consensus_display.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}",
    )
)


print(
    "\n"
    + "=" * 120
)

print(
    "ANALYSIS COMPLETE"
)

print(
    "=" * 120
)


print(
    "\nResults saved to:"
)

print(
    OUTPUT_DIR
)


print(
    "\nMost important manuscript outputs:"
)

print(
    "  1. H10_manuscript_per_class_performance.csv"
)

print(
    "  2. H10_mean_normalized_confusion_matrix_10seeds.png"
)

print(
    "  3. H10_mean_normalized_confusion_matrix_10seeds.svg"
)

print(
    "  4. H10_per_class_summary_10seeds.csv"
)

print(
    "  5. H10_overall_performance_summary.csv"
)

print(
    "  6. H10_consensus_confusion_matrix.png"
)

print(
    "  7. H10_consensus_per_class_metrics.csv"
)

print(
    "\nNo model was retrained."
)

print(
    "All results were calculated directly from the "
    "successful H=10 pooled OOF predictions."
)